# Case Prioritization Predictive

## Problem Framing

**Business question:** Which residents most urgently need staff follow-up in the next 60 days?

**Primary users:** case managers, safehouse leadership

**Decision supported:** Help staff triage limited time toward residents with the strongest short-term follow-up signal.

**Modeling goal:** Prediction. This notebook is judged first on how well it scores new, unseen records rather than on whether a single coefficient is easy to interpret.

**Target / outcome anchor:** Current predictive label: `label_case_prioritization_next_60d`, combining future incidents with coordinated counseling and visitation follow-up signals.

**Submission status:** Artifact-backed predictive pipeline. The repository includes committed model and evaluation artifacts for this track.

Prediction is the right framing here because the organization needs an operational ranking or forecast that can be applied to future records. This notebook therefore prioritizes leakage control, reproducible feature availability at scoring time, and honest out-of-sample validation. Causal interpretation is handled more carefully in the paired explanatory notebook for the same pipeline.


## Data Acquisition, Preparation & Exploration

**Data source and grain:** This notebook loads `resident_monthly_features`, which contains resident-month snapshots that roll up case history, counseling, education, health, visitation, and safehouse context at a point-in-time grain suitable for forward-looking scoring and explanation.

**Reproducible preparation pipeline:** The code cells below use the shared notebook bootstrap plus `load_notebook_context(...)` so the data loading, joins, cleaning, and feature engineering come from reusable modules under `ml/src/data/`, `ml/src/features/`, and the pipeline-specific implementation for `case_prioritization` rather than from one-off notebook code.

**Exploration focus:** Before trusting model output, review the shared table preview, target availability, missingness patterns, date coverage, outliers, and any fields that appear to encode post-outcome information. The goal is to confirm that the data can answer the business question without leakage.

**Shared references used by this notebook:**
* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-b-notebook-standardization.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`


In [1]:
print("Notebook execution proof: case_prioritization predictive template rendered successfully.")


Notebook execution proof: case_prioritization predictive template rendered successfully.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='case_prioritization',
    dataset_name='resident_monthly_features',
    predictive_impl='case_prioritization',
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


## Modeling & Feature Selection

This notebook is prediction-first, so candidate models should be compared using the saved evaluation bundle and the strongest out-of-sample metric for this target. The preferred model is whichever generalizes best while still being stable enough to deploy in staff workflows.

Feature selection follows four rules in this submission:

1. Keep only fields available at scoring time.
2. Remove identifiers, direct future labels, and post-outcome leakage.
3. Favor features with stable business meaning and tolerable missingness.
4. Use domain reasoning alongside model importance so the final feature set is defensible.

**Committed artifact summary:** The committed best model is `random_forest_classifier` for target `label_case_prioritization_next_60d`, selected on `average_precision` = `0.4293`, using `756` training rows and `715` holdout rows, with holdout positive rate `0.2979`.

**Feature inventory:** 61 engineered inputs are currently stored in `ml/models/case_prioritization/feature_list.json`.


## Evaluation & Interpretation

**Validation discipline:** The code cells below pull the saved metrics JSON, holdout comparison table, and cross-validation summary for this pipeline. Those artifacts are the correct basis for judging model quality, not in-sample fit.

**Business meaning of the score:** A high score means the record looks more likely to match this pipeline's target outcome on unseen future data, so `Help staff triage limited time toward residents with the strongest short-term follow-up signal.` can be prioritized earlier.

**Real-world error consequences:** False negatives matter because they can hide girls, cases, or sites that need early intervention. False positives matter because staff time, bed capacity, and follow-up slots are limited and can be pulled away from higher-need situations.

Operationally, the right threshold depends on the workflow. Teams should read precision, recall, ranking quality, and class balance together before deciding whether the score should drive alerts, queue ordering, or a softer recommendation panel.


In [ ]:
evaluation = load_evaluation_bundle('case_prioritization')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


In [ ]:
summarize_frame(holdout_comparison)


In [ ]:
summarize_frame(cv_summary)


## Causal and Relationship Analysis

This notebook uses predictive relationships, not causal identification. Strong features matter because they help separate future outcomes on unseen data, but they do **not** prove that changing the feature will change the outcome.

**Most impactful committed features:** The strongest committed model signals currently surfaced in the repo are `initial risk level Low`, `initial risk level Critical`, `avg nutrition score 90d`, `process recent 90d count`, `process progress flags 90d`. These are useful for prioritization and investigation, but they still represent association rather than proof of causation.

**Decision guidance:** Use these high-signal features to focus staff review, choose follow-up questions, and prioritize intervention or outreach. When leadership wants to argue that a driver should become policy, treatment, or strategy, they should pair this notebook with the explanatory companion and with domain judgment about omitted variables, timing, and selection bias.


## Deployment Notes

**Canonical widgets for the web app:**
* `ranked_table_widget`
* `recommendation_panel`
* `insight_summary_card`

**Submission deployment notes:**
* Use ranked caseload tables during shift handoff and case conference preparation.
* Pair the score with a short recommendation panel describing the strongest urgency drivers.

**Artifact locations referenced by this notebook:**
* `ml/models/case_prioritization/metrics.json`
* `ml/models/case_prioritization/feature_list.json`
* `ml/models/case_prioritization/explainability.csv`
* `ml/reports/evaluation/case_prioritization_metrics.json`

**Endpoint pattern used in the app layer when this track is scored:**
* `POST /ml/predict/case_prioritization`
* `POST /ml/score-batch/case_prioritization`
